<a href="https://colab.research.google.com/github/alexpanayi474/DSC511-MovieLens-Project/blob/main/DSC_511_MovieLens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DSC 511 PROJECT**
- Describe Data:
- Describe the objective:

Load Data and Relevant Libraries

In [ ]:
# Give access to google colab to have access to your google drive, in order to be able to read the datasets
from google.colab import drive
drive.mount('/content/gdrive')
google_drive_path = "/content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/"

In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import *
from pyspark.ml.linalg import Vectors, SparseVector
from pyspark.ml.clustering import LDA
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from textblob import TextBlob
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import pandas as pd

# create a spark session and import graphframes
spark = SparkSession.builder.appName("DSC_511_PROJECT").config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.4-s_2.12").getOrCreate()

**EDA**


In [ ]:
from graphframes import *

ratings_df = spark.read.csv(google_drive_path + 'ratings.csv', header=True, inferSchema=True)
ratings_df.printSchema()

movies_df = spark.read.csv(google_drive_path + 'movies.csv', header=True, inferSchema=True)
movies_df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



In [11]:
ratings_df.count()

33832162

In [12]:
movies_df.count()

86537

In [8]:
ratings_df.show(5)
movies_df.show(5)

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|      1|   4.0|1225734739|
|     1|    110|   4.0|1225865086|
|     1|    158|   4.0|1225733503|
|     1|    260|   4.5|1225735204|
|     1|    356|   5.0|1225735119|
+------+-------+------+----------+
only showing top 5 rows
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


In [9]:
from pyspark.sql.functions import col, from_unixtime, to_timestamp

ratings_clean = (
    ratings_df.dropna(subset=["userId","movieId","rating","timestamp"])
           .withColumn("ts", to_timestamp(from_unixtime(col("timestamp"))))
)

movies_clean = movies_df.dropna(subset=["movieId","title"])

In [10]:
df = ratings_clean.join(movies_clean, on="movieId", how="inner")
df.select("userId","movieId","rating","ts","title","genres").show(5, truncate=False)


+------+-------+------+-------------------+-----------------------------------------+-------------------------------------------+
|userId|movieId|rating|ts                 |title                                    |genres                                     |
+------+-------+------+-------------------+-----------------------------------------+-------------------------------------------+
|1     |1      |4.0   |2008-11-03 17:52:19|Toy Story (1995)                         |Adventure|Animation|Children|Comedy|Fantasy|
|1     |110    |4.0   |2008-11-05 06:04:46|Braveheart (1995)                        |Action|Drama|War                           |
|1     |158    |4.0   |2008-11-03 17:31:43|Casper (1995)                            |Adventure|Children                         |
|1     |260    |4.5   |2008-11-03 18:00:04|Star Wars: Episode IV - A New Hope (1977)|Action|Adventure|Sci-Fi                    |
|1     |356    |5.0   |2008-11-03 17:58:39|Forrest Gump (1994)                      |Comed